# Random Forest Regression with Temporal Features

In [1]:
###  1. SETUP & DATA LOADING 
## Load Data, import required libraries, sklearn and matplot
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt

##Read data file
filename_read = os.path.join("energydata_complete.csv")
data = pd.read_csv(filename_read)

# print(data.head())
print(data.shape)
data.info()

(19735, 29)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19735 entries, 0 to 19734
Data columns (total 29 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         19735 non-null  object 
 1   Appliances   19735 non-null  int64  
 2   lights       19735 non-null  int64  
 3   T1           19735 non-null  float64
 4   RH_1         19735 non-null  float64
 5   T2           19735 non-null  float64
 6   RH_2         19735 non-null  float64
 7   T3           19735 non-null  float64
 8   RH_3         19735 non-null  float64
 9   T4           19735 non-null  float64
 10  RH_4         19735 non-null  float64
 11  T5           19735 non-null  float64
 12  RH_5         19735 non-null  float64
 13  T6           19735 non-null  float64
 14  RH_6         19735 non-null  float64
 15  T7           19735 non-null  float64
 16  RH_7         19735 non-null  float64
 17  T8           19735 non-null  float64
 18  RH_8         19735 non-null  float

# Temporal feature engineering (add time-based predictors)

In [2]:
# Extract useful numeric time features from the timestamp
# (Random Forest cannot use raw datetime strings directly)
data['date'] = pd.to_datetime(data['date'])
data['hour'] = data['date'].dt.hour
data['day_of_week'] = data['date'].dt.dayofweek
data['month'] = data['date'].dt.month


Previous code continued

In [3]:
### 2. PARTINIONING TRAINING DATA AND TEST DATA

## SET target column, column 'Appliances'
target = "Appliances"

## DROP date column, string
## TARGET from X
X = data.drop(columns=["date", target]).values
y = data[target].values

## Train:Test split ( 25% test size ) + reproducable random state, 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)


In [4]:
# - 3. BASELINE RANDOM FOREST MODEL -
# Baseline defined using BOTH hyperparameters:
##  n_estimators = 1 OR single tree
## max_depth = BASE_DEPTH (controls tree complexity)

BASE_DEPTH = 5  # baseline depth choice (simple, not too deep)

baseline_rf = RandomForestRegressor(
    n_estimators=1,
    max_depth=BASE_DEPTH,
    random_state=42,
    n_jobs=-1
)

baseline_rf.fit(X_train, y_train)
baseline_pred = baseline_rf.predict(X_test)

baseline_r2 = r2_score(y_test, baseline_pred)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print("BASELINE MODEL RESULTS")
print(f"Trees: 1 | max_depth: {BASE_DEPTH}")
print(f"R2:  {baseline_r2:.4f}")
print(f"MAE: {baseline_mae:.3f}")
print("-" * 40)

BASELINE MODEL RESULTS
Trees: 1 | max_depth: 5
R2:  0.1613
MAE: 49.560
----------------------------------------


In [5]:
### - 4. HYPERPARAMETER TUNING -
## 4.1 Tuning n_estimator (fixed max_depth) 

r2_list = []
mae_list = []
nums = []

rf = RandomForestRegressor(
    n_estimators=1,
    # Start with
    max_depth=BASE_DEPTH,
    random_state=42,
    n_jobs=-1,
    warm_start=True
)

for i in range(1, 129):
    rf.set_params(n_estimators=i)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)
    r2_list.append(r2_score(y_test, y_pred))
    mae_list.append(mean_absolute_error(y_test, y_pred))
    nums.append(i)

## 4.2 Initiliasing Successive Model (with fixed max_depth)

best_index = np.argmin(mae_list)
best_trees = nums[best_index]
best_mae = mae_list[best_index]
best_r2 = r2_list[best_index]

print("BEST n_estimators (depth fixed)")
print(f"Trees: {best_trees} | max_depth: {BASE_DEPTH}")
print(f"R2:  {best_r2:.4f}")
print(f"MAE: {best_mae:.3f}")
print("-" * 40)


BEST n_estimators (depth fixed)
Trees: 58 | max_depth: 5
R2:  0.2017
MAE: 49.013
----------------------------------------


In [6]:
## 4.3 Tuning max_depth (n_estimators fixed) 

depths = list(range(2, 21))
depth_mae = []
depth_r2 = []

for d in depths:
    rf_depth = RandomForestRegressor(
        n_estimators=best_trees,
        max_depth=d,
        random_state=42,
        n_jobs=-1,
        warm_start=True
    )
    rf_depth.fit(X_train, y_train)
    y_pred = rf_depth.predict(X_test)

    depth_r2.append(r2_score(y_test, y_pred))
    depth_mae.append(mean_absolute_error(y_test, y_pred))

## 4.4 Initiliasing Successive Model (with fixed n_estimators)
best_depth_idx = np.argmin(depth_mae)
best_depth = depths[best_depth_idx]
best_mae_depth = depth_mae[best_depth_idx]
best_r2_depth = depth_r2[best_depth_idx]

print("BEST max_depth (trees fixed)")
print(f"Trees: {best_trees} | max_depth: {best_depth}")
print(f"R2:  {best_r2_depth:.4f}")
print(f"MAE: {best_mae_depth:.3f}")
print("-" * 40)

BEST max_depth (trees fixed)
Trees: 58 | max_depth: 20
R2:  0.5173
MAE: 33.549
----------------------------------------


In [7]:
# - 5. FINAL TUNED MODEL (n_estimators AND max_depth) -

final_rf = RandomForestRegressor(
    n_estimators=best_trees,
    max_depth=best_depth,
    random_state=42,
    n_jobs=-1,
    warm_start=True
)
final_rf.fit(X_train, y_train)
final_pred = final_rf.predict(X_test)

final_r2 = r2_score(y_test, final_pred)
final_mae = mean_absolute_error(y_test, final_pred)

print("FINAL MODEL RESULTS")
print(f"Trees: {best_trees} | max_depth: {best_depth}")
print(f"R2:  {final_r2:.4f}")
print(f"MAE: {final_mae:.3f}")
print("-" * 40)

print("IMPROVEMENT OVER BASELINE")
print(f"Δ R2:  {final_r2 - baseline_r2:.4f}")
print(f"Δ MAE: {baseline_mae - final_mae:.3f}")
print("-" * 40)



FINAL MODEL RESULTS
Trees: 58 | max_depth: 20
R2:  0.5173
MAE: 33.549
----------------------------------------
IMPROVEMENT OVER BASELINE
Δ R2:  0.3560
Δ MAE: 16.011
----------------------------------------


#### Plotting data with the configurated Random Forest Regression models, variating hyperparamters.

In [ ]:
### - 6. PERFORMANCE VISUALISATION -

## 6.1 MAE vs n_estimators (depth fixed)
plt.figure(figsize=(15, 6))
plt.plot(nums, mae_list)
plt.xlabel("Number of Trees (n_estimators)")
plt.ylabel("MAE (lower is better)")
plt.title(f"MAE vs n_estimators (max_depth = {BASE_DEPTH})")
plt.show()

## 6.3 MAE vs max_depth (trees fixed)
plt.figure(figsize=(15, 6))
plt.plot(depths, depth_mae)
plt.xlabel("max_depth")
plt.ylabel("MAE (lower is better)")
plt.title(f"MAE vs max_depth (n_estimators = {best_trees})")
plt.show()


In [ ]:
## 6.2 R2 vs n_estimators (depth fixed)
plt.figure(figsize=(15, 6))
plt.plot(nums, r2_list)
plt.xlabel("Number of Trees (n_estimators)")
plt.ylabel("R² (higher is better)")
plt.title(f"R² vs n_estimators (max_depth = {BASE_DEPTH})")
plt.show()


## 6.4 R2 vs max_depth (trees fixed)
plt.figure(figsize=(15, 6))
plt.plot(depths, depth_r2)
plt.xlabel("max_depth")
plt.ylabel("R² (higher is better)")
plt.title(f"R² vs max_depth (n_estimators = {best_trees})")
plt.show()

In [ ]:
### - 7. OPTIONAL DATA DERIVATION: BINARY HIGH ENERGY USAGE -

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Define binary target: high energy usage if above median
median_val = data["Appliances"].median()
data["high_energy"] = (data["Appliances"] > median_val).astype(int)

#binary target
X_bin = data.drop(columns=["date", "Appliances", "high_energy"]).values
y_bin = data["high_energy"].values

X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_bin, y_bin, test_size=0.25, random_state=42
)


accuracy_data = []
tree_data = []
nums2 = []


rf2 = RandomForestClassifier(
    n_estimators=1,
    criterion="entropy",
    random_state=42,
    n_jobs=-1,
    warm_start = True 
)

for i in range(1, 129):
    rf2.set_params(n_estimators=i)
    rf2.fit(X_train_bin, y_train_bin)
    y_pred_bin = rf2.predict(X_test_bin)

    acc = accuracy_score(y_test_bin, y_pred_bin)
    nums2.append(i)
    accuracy_data.append(acc)

    print(f"Trees: {i:3d} | Accuracy: {acc:.5f}")

plt.figure(figsize=(15, 6))
plt.plot(nums2, accuracy_data)
plt.xlabel("Number of Trees (n_estimators)")
plt.ylabel("Accuracy")
plt.title("Random Forest Classifier Accuracy vs Number of Trees")
plt.show()